# Day 1 - Lab 3: Independent Challenge
## AI and Machine Learning Fundamentals
### Electricity Consumption Forecasting Dataset

**Goal:** Build a full time-series forecasting pipeline from scratch.

**Minimum deliverables:**
1. Clean and prepare datetime data
2. Build lag/date features
3. Train at least 2 forecasting models
4. Evaluate with MAE / RMSE / MAPE
5. Plot actual vs forecast

Good luck!


## Setup

In [2]:
# Run this cell once to install any missing libraries.
# Remove the # at the start of the line below, run it, then add the # back.
# !pip install mlxtend prophet xgboost tensorflow scikit-learn pandas matplotlib seaborn

print("Libraries ready!")


Libraries ready!


In [3]:
# --- Standard library ---
import os
import warnings
import urllib.request

warnings.filterwarnings('ignore')   # hide unimportant warning messages

# --- Data and maths ---
import numpy as np
import pandas as pd

# --- Plotting ---
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')

# --- Preprocessing ---
from sklearn.preprocessing import StandardScaler, LabelEncoder

# --- Models ---
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# --- Utilities ---
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, classification_report,
    mean_squared_error, mean_absolute_error, r2_score
)

print("All imports successful!")


All imports successful!


## Data: already loaded for you

In [4]:
# Electricity forecasting dataset
IS_KAGGLE = os.path.exists('/kaggle/working')

if IS_KAGGLE:
    # Kaggle dataset link:
    # https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption
    candidate_paths = [
        '/kaggle/input/hourly-energy-consumption/PJME_hourly.csv',
        '/kaggle/input/hourly-energy-consumption/AEP_hourly.csv',
        '/kaggle/input/hourly-energy-consumption/COMED_hourly.csv',
    ]
    DATA_PATH = None
    for p in candidate_paths:
        if os.path.exists(p):
            DATA_PATH = p
            break

    if DATA_PATH is None:
        # Fallback to a public daily electricity dataset
        DATA_PATH = 'opsd_germany_daily.csv'
        if not os.path.exists(DATA_PATH):
            urllib.request.urlretrieve(
                'https://raw.githubusercontent.com/jenfly/opsd/master/opsd_germany_daily.csv',
                DATA_PATH
            )
        df_raw = pd.read_csv(DATA_PATH)
        df_raw = df_raw[['Date', 'Consumption']].dropna().copy()
        df_raw.columns = ['Datetime', 'Consumption']
    else:
        df_raw = pd.read_csv(DATA_PATH)
        # Hourly Kaggle files have two columns
        df_raw.columns = ['Datetime', 'Consumption']
else:
    DATA_PATH = 'opsd_germany_daily.csv'
    if not os.path.exists(DATA_PATH):
        urllib.request.urlretrieve(
            'https://raw.githubusercontent.com/jenfly/opsd/master/opsd_germany_daily.csv',
            DATA_PATH
        )
    df_raw = pd.read_csv(DATA_PATH)
    df_raw = df_raw[['Date', 'Consumption']].dropna().copy()
    df_raw.columns = ['Datetime', 'Consumption']

print('Loaded from:', DATA_PATH)
print(df_raw.shape)
df_raw.head()


Loaded from: opsd_germany_daily.csv
(4383, 2)


,Datetime,Consumption
0,2006-01-01,1069.184
1,2006-01-02,1380.521
2,2006-01-03,1442.533
3,2006-01-04,1457.217
4,2006-01-05,1477.131


## Your Tasks

All imports and data loading are done. Implement the forecasting pipeline below.

---

### Step 1: EDA
- Print the shape, column types, and missing value counts
- Convert `Datetime` to datetime and set it as index
- Plot the full time series

### Step 2: Data Cleaning (7 steps)
1. Sort by datetime and remove duplicates
2. Fill missing `Consumption` values
3. Resample if needed (daily or hourly)
4. Create lag features (`lag_1`, `lag_7`)
5. Create rolling mean features
6. Add date features (`dayofweek`, `month`)
7. Split train/test by time (no shuffle)

### Step 3: Forecasting Models
- Train at least 2 models (example: Linear Regression, RandomForestRegressor)
- Predict future values on test set

### Step 4: Evaluation
- Compute MAE, RMSE, and MAPE
- Plot Actual vs Predicted

### Step 5: Advanced
- Try one advanced method (XGBoost, Prophet, or LSTM)
- Compare all models in a final table (sorted by RMSE)

---
**Bonus:** Forecast the next 30 periods and plot them.


In [5]:
# Your code starts here
df = df_raw.copy()
